# 04 · The Batch Pipeline

> **Module 6 · Batch Pipelines · Lectures 6.1–6.6**

In this notebook you will:

- Build the batch revenue pipeline step by step from the course project modules
- Measure the filter-early optimisation instead of taking it on faith
- Route malformed rows to a dead-letter queue
- Prove the pipeline is idempotent by running it twice and comparing
- Parameterise a backfill so a historical slice can be reprocessed on demand

---

## 1. Setup

Everything below runs on the **Direct Runner**: no GCP project, no credentials,
no bill. The same code runs unchanged on Dataflow — only the `--runner` flag and
the path prefixes change.

The project modules live in `project/`. We put that directory on `sys.path` so
`from transforms import ...` works exactly as it does in the scripts.


In [ ]:
import os
import sys
import subprocess
import tempfile
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = COURSE_ROOT.parent
PROJECT_DIR = COURSE_ROOT / "project"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions, TypeOptions

print(f"apache-beam {beam.__version__}")
print(f"course root : {COURSE_ROOT}")
print(f"project dir : {PROJECT_DIR}")

# Make sure the sample data exists — regenerate it only when it is missing.
ORDERS_CSV = COURSE_ROOT / "sample_data" / "orders.csv"
if not ORDERS_CSV.exists():
    subprocess.run(
        [sys.executable, "resources/sample_data.py", "--output-dir", "./sample_data"],
        check=True,
        cwd=str(COURSE_ROOT),
    )

print(f"orders.csv : {ORDERS_CSV} ({ORDERS_CSV.stat().st_size:,} bytes)")


## 2. The pipeline shape, and how a row gets parsed

The batch pipeline is a straight line of transforms. Read it as a sentence:

| Step | Transform | What it does |
|------|-----------|--------------|
| 1 | `ReadFromText` | Read the CSV lines off disk |
| 2 | `ParDo(ParseOrderFn())` | Parse and validate each line |
| 3 | `Filter` | Keep only the statuses we count |
| 4 | `Map` | Key each order by region |
| 5 | `CombinePerKey(RevenueByRegionFn())` | Sum count and revenue per region |
| 6 | `Map(to_summary_row)` | Flatten `(region, summary)` into a row |
| 7 | `WriteToText` | Write the CSV output |

Two things are worth noticing before we write a line of code.

**The `CombineFn` is the only stateful step.** Everything else is a per-element
map or filter. That is what makes the pipeline scale: the expensive part runs
per key, in a combiner that Beam can distribute.

**The status filter sits *before* the combine.** Move it after and every
cancelled order still has to travel through the shuffle. We will measure the
difference in section 3.

`ParseOrderFn` is a `DoFn` that parses a CSV line into a dict and **skips**
malformed rows rather than failing the bundle. That is the right default for a
batch pipeline over a file you do not control: one bad line should not cost you
the other nine million.

We can see the parser's decisions without a runner at all, because the pure
helpers behind it are ordinary Python.


In [ ]:
from transforms import (
    ParseOrderFn,
    RevenueByRegionFn,
    is_valid_order,
    parse_order_line,
    try_parse_order_line,
)

# A tiny, readable slice of the real sample file.
with ORDERS_CSV.open(encoding="utf-8") as fh:
    header = fh.readline().rstrip("\n")
    first_rows = [fh.readline().rstrip("\n") for _ in range(3)]

print("header:", header)
for row in first_rows:
    print("row   :", row)
print()

parsed = parse_order_line(first_rows[0])
print("parsed type :", type(parsed).__name__)
for key, value in parsed.items():
    print(f"  {key:<12} = {value!r}")
print()
print("valid?      :", is_valid_order(parsed))
print()

# The malformed rows the sample generator writes on purpose.
MALFORMED = COURSE_ROOT / "sample_data" / "malformed_orders.csv"
if MALFORMED.exists():
    for line in MALFORMED.read_text(encoding="utf-8").splitlines()[:3]:
        row = try_parse_order_line(line)
        if row is None:
            verdict = "rejected (unparseable)"
        elif not is_valid_order(row):
            verdict = "rejected (invalid)"
        else:
            verdict = "accepted"
        print(f"  {verdict:<22} {line[:66]}")


## 3. The filter-early optimisation, measured

The claim: filtering invalid and unwanted rows **before** the shuffle is
cheaper than filtering after it.

We can measure this locally. The metric that matters is **how many elements
reach the combiner**, because the combiner is the step that shuffles data
across workers. Fewer elements into the shuffle means less network traffic,
less disk spill, and a shorter job.

Let's count both ways.


In [ ]:
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.util import assert_that, equal_to

ORDER_LINES = [
    "o1,c1,2026-01-15T10:00:00+00:00,EMEA,web,EUR,complete,100.00,p1",
    "o2,c2,2026-01-15T11:00:00+00:00,EMEA,web,USD,complete,200.00,p2",
    "o3,c3,2026-01-15T12:00:00+00:00,AMER,web,EUR,complete,50.00,p3",
    "o4,c4,2026-01-15T13:00:00+00:00,AMER,web,EUR,cancelled,999.00,p4",
    "o5,c5,2026-01-15T14:00:00+00:00,EMEA,web,GBP,complete,100.00,p5",
    "o6,c6,2026-01-15T15:00:00+00:00,EMEA,web,EUR,cancelled,500.00,p6",
    "o7,c7,2026-01-15T16:00:00+00:00,AMER,web,EUR,refunded,300.00,p7",
]

WORK = Path(tempfile.mkdtemp(prefix="dataflow_nb04_"))
TEST_OPTS = PipelineOptions(["--runner=BundleBasedDirectRunner"])


def _parsed(lines, pipeline):
    """Parse raw CSV lines into order dicts."""
    return (
        pipeline
        | "Create" >> beam.Create(lines)
        | "Parse" >> beam.ParDo(ParseOrderFn())
    )


def _read_pairs(path):
    """Read (key, value) pairs back from a text file written by WriteToText."""
    rows = []
    for f in sorted(path.parent.glob(path.name + "*")):
        for line in f.read_text(encoding="utf-8").splitlines():
            if line.strip():
                key, _, value = line.partition(",")
                rows.append((key, int(value)))
    return sorted(rows)


# --- Path A: filter FIRST (what the course pipeline does) ---------------
EARLY_OUT = WORK / "early_counts"
with TestPipeline(options=TEST_OPTS) as p:
    filtered_first = (
        _parsed(ORDER_LINES, p)
        | "FilterEarly" >> beam.Filter(lambda order: order["status"] == "complete")
        | "KeyForCount" >> beam.Map(lambda order: (order["region"], 1))
        | "CountEarly" >> beam.combiners.Count.PerKey()
    )
    assert_that(filtered_first, equal_to([("AMER", 1), ("EMEA", 3)]))
    (
        filtered_first
        | "FormatEarly" >> beam.Map(lambda kv: f"{kv[0]},{kv[1]}")
        | "WriteEarly" >> beam.io.WriteToText(str(EARLY_OUT), num_shards=1, shard_name_template="")
    )

early_rows = _read_pairs(EARLY_OUT)

# --- Path B: filter AFTER the combine -----------------------------------
LATE_OUT = WORK / "late_counts"
with TestPipeline(options=TEST_OPTS) as p:
    combined_first = (
        _parsed(ORDER_LINES, p)
        | "KeyByRegion" >> beam.Map(lambda order: (order["region"], order))
        | "CombineFirst" >> beam.CombinePerKey(RevenueByRegionFn())
    )
    filtered_late = (
        combined_first
        | "FilterLate" >> beam.Filter(lambda kv: kv[1]["order_count"] > 0)
        | "ToCount" >> beam.Map(lambda kv: (kv[0], kv[1]["order_count"]))
    )
    assert_that(filtered_late, equal_to([("AMER", 3), ("EMEA", 4)]))
    (
        filtered_late
        | "FormatLate" >> beam.Map(lambda kv: f"{kv[0]},{kv[1]}")
        | "WriteLate" >> beam.io.WriteToText(str(LATE_OUT), num_shards=1, shard_name_template="")
    )

late_rows = _read_pairs(LATE_OUT)

print("Filter FIRST  -> elements into the shuffle:", sum(count for _, count in early_rows), early_rows)
print("Filter AFTER  -> elements into the shuffle:", sum(count for _, count in late_rows), late_rows)
print()
print("The second pipeline counted the cancelled and refunded orders too")
print("(AMER=3, EMEA=4 instead of AMER=1, EMEA=3).")
print("Filtering after the combine cannot undo it: the shuffle has already happened,")
print("and the combiner has already added those amounts to the regional totals.")


The result is not just "fewer elements". It is **a different answer**.

Filtering after the combine gives `AMER = 3` and `EMEA = 4`, because the
cancelled and refunded orders were already folded into the accumulator.
Filtering first gives `AMER = 1` and `EMEA = 3` — the number the business
actually asked for.

That is the real lesson: **filters belong before the stateful step**. A filter
after a `CombinePerKey` is not just slower, it is usually *wrong*.

## 4. The dead-letter queue

`ParseOrderFn` logs a warning and skips a malformed row. That is fine for a
batch job over a file you can re-read. It is **not** fine for a pipeline that
must not lose data — a silently dropped row is a row nobody will ever know
about.

The production pattern is a **dead-letter queue**: every rejected row is
written somewhere durable so it can be inspected, fixed, and replayed.

Beam 2.77 has no `FilterInto` transform, so we build the same thing out of
two steps that every Beam version supports:

1. `Map` each element to a **tagged** tuple: `("good", value)` or `("bad", value)`
2. `Filter` on the tag to split the tagged PCollection into two streams

This is the canonical Beam DLQ idiom, and it composes with any sink. In
production the DLQ sink is the part that matters: a DLQ written to the same
filesystem as the good output, with the same retention, is a DLQ nobody ever
looks at. It needs its own location, its own alerting, and a re-drive job.


In [ ]:
def _tag_parse_result(line):
    """Parse a line and tag it as good or bad.

    Args:
        line: One raw CSV line.

    Returns:
        Either ("good", order_dict) or ("bad", (line, reason)).
    """
    row = try_parse_order_line(line)
    if row is None:
        return "bad", (line, "unparseable: wrong field count or non-numeric amount")
    if not is_valid_order(row):
        return "bad", (line, "invalid: failed a business rule")
    return "good", row


def _bad_reason(entry):
    """Format a dead-letter entry as one text line."""
    line, reason = entry
    return f"{reason}\t{line}"


def _format_order(order):
    """Format one accepted order as a CSV line."""
    return (
        f"{order['order_id']},{order['region']},"
        f"{order['status']},{order['amount']},{order['currency']}"
    )


DLQ_LINES = ORDER_LINES + [
    "not-an-order,missing-fields,garbage",
    "o99999998,c000002,2026-01-01T00:00:00+00:00,UNKNOWN_REGION,web,EUR,complete,100.00,p0002",
]

GOOD_OUT = WORK / "good.csv"
DLQ_OUT = WORK / "dlq.csv"

with TestPipeline(options=TEST_OPTS) as p:
    tagged = (
        p
        | "ReadLines" >> beam.Create(DLQ_LINES)
        | "TagLines" >> beam.Map(_tag_parse_result)
    )
    good = (
        tagged
        | "KeepGood" >> beam.Filter(lambda kv: kv[0] == "good")
        | "DropGoodTag" >> beam.Map(lambda kv: kv[1])
    )
    bad = (
        tagged
        | "KeepBad" >> beam.Filter(lambda kv: kv[0] == "bad")
        | "DropBadTag" >> beam.Map(lambda kv: kv[1])
    )

    assert_that(
        good,
        equal_to([parse_order_line(line) for line in ORDER_LINES]),
        label="assert_good",
    )
    assert_that(
        bad,
        equal_to([
            (
                "not-an-order,missing-fields,garbage",
                "unparseable: wrong field count or non-numeric amount",
            ),
            (
                "o99999998,c000002,2026-01-01T00:00:00+00:00,UNKNOWN_REGION,web,EUR,complete,100.00,p0002",
                "invalid: failed a business rule",
            ),
        ]),
        label="assert_bad",
    )

    (
        good
        | "FormatGood" >> beam.Map(_format_order)
        | "WriteGood" >> beam.io.WriteToText(str(GOOD_OUT), num_shards=1, shard_name_template="")
    )
    (
        bad
        | "FormatBad" >> beam.Map(_bad_reason)
        | "WriteDlq" >> beam.io.WriteToText(str(DLQ_OUT), num_shards=1, shard_name_template="")
    )

print("good output (order_id, region, status, amount, currency):")
print(GOOD_OUT.read_text(encoding="utf-8"))
print("dlq output (reason, raw line):")
print(DLQ_OUT.read_text(encoding="utf-8"))


## 5. A deliberate mistake: duplicate transform labels

Beam identifies every transform in a pipeline by a **label**. Two transforms
with the same label in the same pipeline is a hard error — Beam has to be able
to name each step in its job graph.

Let's trigger the real error and read it.


In [ ]:
try:
    with beam.Pipeline(options=TEST_OPTS) as p:
        (
            p
            | "Read" >> beam.Create(ORDER_LINES)
            | "Parse" >> beam.ParDo(ParseOrderFn())
            | "Parse" >> beam.Map(lambda order: order["region"])
        )
except Exception as exc:
    print("ERROR TYPE:", type(exc).__name__)
    print("ERROR MSG :", exc)


The message names the colliding label and the position of the *second* one:

```
RuntimeError: A transform with label "[6]: Parse" already exists in the pipeline.
```

The `[6]` is the transform's index in the pipeline. It is more helpful than a
bare "name already used", but it still does not point at a line of your code.
The practical rule is: **make labels unique per pipeline by construction**, not
by review. A prefix per stage (`ParseOrders`, `ParseEvents`) or a helper that
appends a suffix is the cheap fix.

Note also the scope: labels are unique **per pipeline**, not per codebase. Two
separate test methods may each use a transform called `"Create"` — that is
fine, they are two different pipelines.

## 6. Idempotency: run it twice, compare

A batch pipeline should be **idempotent** — running it twice over the same
input should produce the same answer, and re-running it should not corrupt
what is already there.

Two things can break that:

1. **Accumulating state across runs.** Beam pipelines are stateless between
   runs, so this is usually safe — but only if you did not write your own
   stateful side file.
2. **A non-idempotent sink.** A sink that appends will append twice. Dataflow's
   BigQuery sink defaults to `WRITE_TRUNCATE`, which is safe for a full rebuild
   but wrong for an incremental load.

Let's prove the pipeline is deterministic by running it twice and diffing the
bytes.

Then we cross-validate the Beam result against the **pure reference
implementation** in `transforms.build_revenue_summary`. This is the single most
valuable test you can have in a batch pipeline, and it costs nothing. The pure
function has no runner, no shuffle, and no coders — it is just a loop. If the
two agree, the Beam plumbing (parsing, encoding, grouping, combining) is
correct. If they disagree, you know the bug is in the Beam layer, not in the
business logic.


In [ ]:
from batch_pipeline import build_pipeline


def run_batch(output_path):
    """Run the course batch pipeline over the sample orders."""
    with beam.Pipeline(options=TEST_OPTS) as p:
        build_pipeline(
            p,
            input_path=str(ORDERS_CSV),
            output_path=str(output_path),
            statuses=("complete",),
        )
    # The sink writes one sharded file; concatenate whatever landed.
    prefix = Path(output_path).name
    files = sorted(f for f in Path(output_path).parent.glob(prefix + "*") if f.is_file())
    return "".join(f.read_text(encoding="utf-8") for f in files)


RUN_A = WORK / "run_a"
RUN_B = WORK / "run_b"
result_a = run_batch(RUN_A)
result_b = run_batch(RUN_B)

print("--- run A ---")
print(result_a)
print("--- run B ---")
print(result_b)
print("identical:", result_a == result_b)


In [ ]:
from transforms import build_revenue_summary

# Read the CSV the same way the pipeline does.
raw_rows = []
with ORDERS_CSV.open(encoding="utf-8") as fh:
    fh.readline()  # skip the header
    for line in fh:
        row = try_parse_order_line(line.strip())
        if row is not None:
            raw_rows.append(row)

reference = build_revenue_summary(raw_rows, statuses=("complete",))
print("pure reference implementation:")
for region in sorted(reference):
    bucket = reference[region]
    print(f"  {region:<6} count={bucket['order_count']:<6} revenue_eur={bucket['revenue_eur']}")

print()
print("Beam pipeline output:")
beam_rows = {}
for line in result_a.splitlines()[1:]:
    if not line.strip():
        continue
    region, count, revenue = line.split(",")
    beam_rows[region] = {"order_count": int(count), "revenue_eur": float(revenue)}

mismatches = []
for region in sorted(reference):
    ref = reference[region]
    got = beam_rows[region]
    ok = (
        ref["order_count"] == got["order_count"]
        and abs(ref["revenue_eur"] - got["revenue_eur"]) <= 0.01
    )
    if not ok:
        mismatches.append(region)
    print(f"  {region:<6} reference={ref}  beam={got}  match={ok}")

print()
print("mismatches:", mismatches or "none")


## 7. Backfill: parameterising the pipeline

A **backfill** reprocesses a historical slice of data — the day the pipeline
was broken, the week a late-arriving feed finally landed, the month you
migrated a schema.

A backfill is not a special program. It is the *same* pipeline with different
parameters. If backfilling needs different code, the pipeline is wrong.

The parameters that matter:

| Parameter | What it selects |
|-----------|-----------------|
| `--input` | which slice of source data |
| `--statuses` | which order statuses count |
| `--output` | a **separate** output location, not the live one |
| `--runner` | `DirectRunner` for a test backfill, `DataflowRunner` for a big one |

That last row matters: a backfill over 200 days of data is a real job, and you
want it on Dataflow with autoscaling — not on your laptop.

Two things a backfill must **not** do:

- **Not write to the live output path.** The backfill output goes to its own
  location; a human decides when to promote it.
- **Not reuse the production job's name.** On Dataflow the job name identifies
  the run in the console and in billing. A backfill named `revenue-summary`
  is indistinguishable from the production job.

The same discipline applies to BigQuery: a backfill writes to a staging table,
and you `MERGE` or swap it in once it is verified.


In [ ]:
from datetime import date

TARGET_DATE = date(2026, 1, 15)


def slice_orders_for_day(target, source_csv):
    """Write the orders for one day to its own CSV file.

    A real backfill would let the source do the filtering (a BigQuery
    partition predicate, a date-prefixed path). Doing it here keeps the
    notebook honest about what a backfill *is*: the same pipeline, a
    different slice of input.

    Args:
        target: The day to slice.
        source_csv: The full orders file.

    Returns:
        The path to the sliced file.
    """
    out = WORK / f"orders_{target.isoformat()}.csv"
    prefix = target.isoformat()
    with source_csv.open(encoding="utf-8") as fh, out.open("w", encoding="utf-8") as out_fh:
        out_fh.write(fh.readline())
        for line in fh:
            if prefix in line:
                out_fh.write(line)
    return out


backfill_input = slice_orders_for_day(TARGET_DATE, ORDERS_CSV)
backfill_output = WORK / f"backfill_{TARGET_DATE.isoformat()}"

with beam.Pipeline(options=TEST_OPTS) as p:
    build_pipeline(
        p,
        input_path=str(backfill_input),
        output_path=str(backfill_output),
        statuses=("complete", "active"),
    )

backfill_files = sorted(
    f for f in WORK.glob(f"backfill_{TARGET_DATE.isoformat()}*") if f.is_file()
)
backfill_text = "".join(f.read_text(encoding="utf-8") for f in backfill_files)
source_rows = len(backfill_input.read_text(encoding="utf-8").splitlines()) - 1
print(f"backfill for {TARGET_DATE} ({source_rows} source rows)")
print(backfill_text)


## 8. Running it on Dataflow (optional, needs credentials)

This cell is **gated**. Without a project it prints a message and does nothing,
so the notebook runs cleanly on a machine with no GCP account.

When you do have a project, the only difference between the local run and the
cloud run is the runner and the path prefixes.

| Concern | Notebook version | Production version |
|---------|------------------|--------------------|
| Bad rows | Logged and skipped | Written to a DLQ with alerting and a re-drive job |
| Output | One CSV file | BigQuery table, partitioned by business date |
| Idempotency | Trivially true (stateless) | `WRITE_TRUNCATE` for full loads, dedup keys for incremental |
| Backfill | Manual slice | Parameterised job with its own output prefix |
| Sizing | Local, one worker | Autoscaled, with the worker count set from the input size |
| Observability | `print()` | `beam.metrics` counters for rejected rows, plus Dataflow monitoring |


In [ ]:
PROJECT_ID = os.environ.get("PROJECT_ID", "")
BUCKET = os.environ.get("BUCKET", "")

if PROJECT_ID and BUCKET:
    options = PipelineOptions(
        [
            "--runner=DataflowRunner",
            f"--project={PROJECT_ID}",
            "--region=us-central1",
            f"--staging_location={BUCKET}/dataflow/staging",
            f"--temp_location={BUCKET}/dataflow/temp",
            f"--job_name=revenue-summary-{TARGET_DATE.isoformat()}",
        ]
    )
    with beam.Pipeline(options=options) as p:
        build_pipeline(
            p,
            input_path=f"{BUCKET}/orders/orders.csv",
            output_path=f"{BUCKET}/summary/{TARGET_DATE.isoformat()}",
            statuses=("complete",),
        )
    print("Dataflow job submitted.")
else:
    print("Skipped — set PROJECT_ID and BUCKET to enable this cell.")
    print("The local run above used the exact same build_pipeline() function.")


## 🔑 Takeaways

- The batch pipeline is a straight line: **read, parse, filter, key, combine, write**, and the `CombineFn` is the only stateful step.
- **Filter before the combine**, not after. Filtering late is not just slower — it produces a *different, wrong* answer, because the unwanted rows are already in the accumulator.
- A **dead-letter queue** turns "we dropped a row" into "we kept the row and know why it was rejected". Beam 2.77 has no `FilterInto`, so the tagged-tuple idiom (`Map` to a tag, then `Filter` on the tag) is the portable way to build one.
- **Transform labels must be unique per pipeline.** Beam's error names the colliding label and its index, but not your line number, so build uniqueness in by construction.
- **Idempotency is a property of the sink, not the pipeline.** Beam pipelines are stateless between runs; appending sinks are what break re-runs.
- **Cross-validate against a pure reference implementation.** A plain-Python `build_revenue_summary` that agrees with the Beam output proves the plumbing is right.
- **A backfill is the same pipeline with different parameters** — different input slice, different output location, and a different runner. Separate code means a wrong pipeline.
